In [ ]:
import os
import math
import time
import torch
import pickle
import numpy as np

import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.autograd as autograd

import torch.nn.functional as functional
import torch.optim.lr_scheduler as lr_scheduler
from utils import *

os.makedirs('./results/many_well_32', exist_ok=True)
torch.set_default_dtype(torch.float32) 
_ = torch.manual_seed(0)

In [ ]:
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
if torch.cuda.is_available():
    torch_device = 'cuda'
else:
    torch_device = 'cpu'
print(f"TORCH DEVICE: {torch_device}")

In [ ]:
dim = 32
filename = "many_well_dataset_32_110000_samples.pkl"
f = open(filename, 'rb')
dataset = pickle.load(f,encoding="latin1")
test_filename = "many_well_dataset_16_20000_test_samples.pkl"
t = open(test_filename,'rb')
testset = pickle.load(t,encoding="latin1")

In [ ]:
dataset.shape

In [ ]:
batch_size = 512
no_training_samples = 100000
no_validation_samples = 20000
no_test_samples = 20000


train_dataloader = torch.utils.data.DataLoader(dataset, batch_size = batch_size, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(dataset[100000:], batch_size = batch_size, shuffle=False)



In [ ]:
model = RNVP_ND(16,10,(16),256, 1.).to(torch_device)

In [ ]:
PATH = "./results/many_well_32/model_fkl_100000.pt"

In [ ]:
# #For resuming the checkpoint:
# checkpoint = torch.load(PATH)
# model.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [ ]:
optimizer = optim.Adam(model.parameters(),lr= 1.e-5)

In [ ]:
epochs =  100
for i in range(epochs):
    model.train()
    forward_loss = 0.
    NLL = 0.

    
    for x in train_dataloader:
        z_ , logq = model.inverse(x)
        fkl_loss = -torch.mean(logq)  # Forward Kl loss

        kl_loss  = fkl_loss
        optimizer.zero_grad()
        kl_loss.backward()    
        optimizer.step()
        
        forward_loss += fkl_loss.item()*(x.shape[0])
        
   
    model.eval()
    with torch.no_grad():
        for x in val_dataloader:
            z_, logq = model.inverse(x)
            nll = - torch.mean(logq)
            NLL += nll.item()*(x.shape[0])
            
    mean_fkl = forward_loss/no_training_samples
    mean_nll = NLL/no_validation_samples

    template = 'Epoch {:3d},FKL loss: {:.4f}, Val loss: {:.4f}'
    print(template.format(i+1 , mean_fkl , mean_nll))

    

In [ ]:
no_test_samples  = 20000

model.eval()
z_, logq = model.inverse(testset)
test_nll = - torch.mean(logq)
print(f'Test NLL: {test_nll:.4f}')

In [ ]:
torch.save({'model_state_dict': model.state_dict(),'optimizer_state_dict': optimizer.state_dict(),}, PATH)